# Multimodal Deepfake Dataset EDA

This notebook builds a structured media manifest from the dataset folders and performs exploratory analysis for dataset quality, class balance, metadata distributions, correlations, outliers, and ML readiness.

**Outputs:** interactive charts, summary tables, and optional exported EDA artifacts under `reports/eda_notebook/`.

In [2]:
# Core imports
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)

from eda_media_report import (
    EDAConfig,
    collect_manifest,
    dataset_overview,
    missing_analysis,
    descriptive_statistics,
    outlier_analysis,
    univariate_analysis,
    bivariate_analysis,
    multivariate_analysis,
    correlation_analysis,
    correlation_pairs,
    target_analysis,
    data_quality_checks,
    statistical_tests,
    feature_engineering_insights,
    ml_readiness,
    write_report,
)

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR.parent / 'dock'
OUTPUT_DIR = PROJECT_DIR / 'reports' / 'eda_notebook'
PLOTS_DIR = OUTPUT_DIR / 'plots'
TABLES_DIR = OUTPUT_DIR / 'tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Set metadata_limit=None for full analysis. Use a smaller number while iterating.
config = EDAConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    metadata_limit=None,
    hash_files=False,
    max_pairplot_rows=1500,
    max_plot_categories=20,
)

print('Notebook ready')

Notebook ready


## 1. Build Dataset Manifest

The dataset is media-based, so the first step is to convert folder/file metadata into a structured tabular manifest.

In [ ]:
df = collect_manifest(config)
df.to_csv(OUTPUT_DIR / 'media_manifest.csv', index=False)

display(Markdown(f'**Rows:** {df.shape[0]:,}  |  **Columns:** {df.shape[1]:,}'))
display(df.head(10))

Indexed 500 media files...
Indexed 1,000 media files...
Indexed 1,500 media files...
Indexed 2,000 media files...
Indexed 2,500 media files...
Indexed 3,000 media files...
Indexed 3,500 media files...
Indexed 4,000 media files...
Indexed 4,500 media files...
Indexed 5,000 media files...
Indexed 5,500 media files...
Indexed 6,000 media files...
Indexed 6,500 media files...
Indexed 7,000 media files...
Indexed 7,500 media files...
Indexed 8,000 media files...
Indexed 8,500 media files...
Indexed 9,000 media files...
Indexed 9,500 media files...
Indexed 10,000 media files...
Indexed 10,500 media files...
Indexed 11,000 media files...
Indexed 11,500 media files...
Indexed 12,000 media files...
Indexed 12,500 media files...
Indexed 13,000 media files...
Indexed 13,500 media files...
Indexed 14,000 media files...
Indexed 14,500 media files...
Indexed 15,000 media files...
Indexed 15,500 media files...
Indexed 16,000 media files...
Indexed 16,500 media files...
Indexed 17,000 media files...
I

## 2. Dataset Overview

In [ ]:
overview = dataset_overview(df)

overview_cards = pd.DataFrame({
    'Metric': ['Rows', 'Columns', 'Memory MB', 'Duplicate Records', 'Numeric Columns', 'Categorical Columns'],
    'Value': [
        f"{overview['rows']:,}",
        f"{overview['columns']:,}",
        f"{overview['memory_mb']:.2f}",
        f"{overview['duplicate_records']:,}",
        len(df.select_dtypes(include=[np.number, 'bool']).columns),
        len(df.select_dtypes(include=['object', 'category']).columns),
    ]
})
display(overview_cards)

dtype_table = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'unique_values': df.nunique(dropna=False),
    'missing_percent': df.isna().mean() * 100,
}).sort_values(['dtype', 'unique_values'], ascending=[True, False])
display(dtype_table)

In [ ]:
# Dataset composition overview
composition = (
    df.groupby(['split', 'modality', 'label'], dropna=False)
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending=False)
)
display(composition.head(30))

fig = px.treemap(
    composition,
    path=['split', 'modality', 'label'],
    values='count',
    color='count',
    color_continuous_scale='Tealgrn',
    title='Dataset Composition by Split, Modality, and Label',
)
fig.update_layout(height=620, margin=dict(t=60, l=10, r=10, b=10))
fig.show()

## 3. Missing Value Analysis

In [ ]:
missing = missing_analysis(df, PLOTS_DIR)
display(missing)

missing_plot = missing.reset_index(names='column').query('missing_count > 0')
if missing_plot.empty:
    display(Markdown('No missing values detected.'))
else:
    fig = px.bar(
        missing_plot.head(35),
        x='missing_percent',
        y='column',
        orientation='h',
        title='Missing Values by Column',
        labels={'missing_percent': 'Missing %', 'column': 'Column'},
        color='missing_percent',
        color_continuous_scale='Reds',
    )
    fig.update_layout(height=max(420, len(missing_plot.head(35)) * 22), yaxis={'categoryorder': 'total ascending'})
    fig.show()

In [ ]:
# Missingness heatmap, sampled for readability if needed
sample_for_missing = df.sample(min(len(df), 1500), random_state=42) if len(df) > 1500 else df
missing_matrix = sample_for_missing.isna().astype(int)

fig = px.imshow(
    missing_matrix.T,
    aspect='auto',
    color_continuous_scale=['#13202e', '#f05d5e'],
    title='Missing Value Heatmap (Sampled Rows if Large)',
    labels={'x': 'Record Index', 'y': 'Column', 'color': 'Missing'},
)
fig.update_layout(height=max(520, len(df.columns) * 18), coloraxis_showscale=False)
fig.show()

## 4. Descriptive Statistics

In [ ]:
numeric_summary, cat_tables = descriptive_statistics(df)
numeric_summary.to_csv(TABLES_DIR / 'numeric_summary.csv')
display(numeric_summary.round(4))

cat_summary = pd.DataFrame([
    {
        'column': col,
        'cardinality': df[col].nunique(dropna=True),
        'top_category': table.index[0] if not table.empty else None,
        'top_count': int(table.iloc[0]['count']) if not table.empty else 0,
        'top_percent': float(table.iloc[0]['percent']) if not table.empty else 0,
    }
    for col, table in cat_tables.items()
]).sort_values('cardinality', ascending=False)
display(cat_summary)

## 5. Univariate Analysis

In [ ]:
distribution_notes = univariate_analysis(df, PLOTS_DIR, config, numeric_summary)

display(Markdown('### Distribution Shape Summary'))
display(pd.DataFrame({
    'positively_skewed': pd.Series(distribution_notes['positively_skewed']),
    'negatively_skewed': pd.Series(distribution_notes['negatively_skewed']),
    'near_normal_by_skew': pd.Series(distribution_notes['near_normal_by_skew']),
}))

key_numeric = [c for c in ['file_size_mb', 'duration_s', 'bit_rate', 'fps', 'width', 'height', 'pixels', 'size_per_second_mb'] if c in df.columns]
for col in key_numeric:
    plot_df = df[df[col].notna()].copy()
    if plot_df.empty:
        continue
    fig = px.histogram(
        plot_df,
        x=col,
        color='label' if 'label' in plot_df else None,
        marginal='box',
        nbins=60,
        opacity=0.75,
        title=f'Distribution of {col}',
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.update_layout(height=520, bargap=0.02)
    fig.show()

In [ ]:
for col in ['label', 'split', 'modality', 'extension', 'video_codec', 'audio_codec']:
    if col not in df.columns:
        continue
    counts = df[col].fillna('Missing').value_counts().head(25).reset_index()
    counts.columns = [col, 'count']
    fig = px.bar(
        counts,
        x='count',
        y=col,
        orientation='h',
        title=f'Top Categories: {col}',
        color='count',
        color_continuous_scale='Viridis',
    )
    fig.update_layout(height=max(380, len(counts) * 24), yaxis={'categoryorder': 'total ascending'})
    fig.show()

## 6. Outlier Analysis

In [ ]:
outliers = outlier_analysis(df, PLOTS_DIR)
outliers.to_csv(TABLES_DIR / 'outlier_summary.csv')
display(outliers.round(4))

if not outliers.empty:
    outlier_plot = outliers.reset_index().sort_values('iqr_outlier_percent', ascending=False).head(25)
    fig = px.bar(
        outlier_plot,
        x='iqr_outlier_percent',
        y='column',
        orientation='h',
        title='IQR Outlier Percentage by Feature',
        color='iqr_outlier_percent',
        color_continuous_scale='OrRd',
    )
    fig.update_layout(height=max(420, len(outlier_plot) * 26), yaxis={'categoryorder': 'total ascending'})
    fig.show()

## 7. Bivariate Analysis

In [ ]:
bivariate_analysis(df, PLOTS_DIR, config)

if {'duration_s', 'file_size_mb'}.issubset(df.columns):
    plot_df = df.dropna(subset=['duration_s', 'file_size_mb']).copy()
    fig = px.scatter(
        plot_df,
        x='duration_s',
        y='file_size_mb',
        color='label' if 'label' in plot_df else None,
        symbol='modality' if 'modality' in plot_df else None,
        hover_data=['file_name', 'split', 'extension'],
        title='File Size vs Duration',
        opacity=0.72,
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.update_layout(height=620)
    fig.show()

if {'width', 'height'}.issubset(df.columns):
    plot_df = df.dropna(subset=['width', 'height']).copy()
    fig = px.scatter(
        plot_df,
        x='width',
        y='height',
        color='label' if 'label' in plot_df else None,
        size='file_size_mb' if 'file_size_mb' in plot_df else None,
        hover_data=['file_name', 'duration_s', 'fps'],
        title='Resolution Profile by Label',
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.update_layout(height=620)
    fig.show()

In [ ]:
for col in ['file_size_mb', 'duration_s', 'fps', 'size_per_second_mb']:
    if col not in df.columns or 'label' not in df.columns:
        continue
    plot_df = df[df[col].notna()].copy()
    if plot_df.empty:
        continue
    fig = px.violin(
        plot_df,
        x='label',
        y=col,
        color='label',
        box=True,
        points='outliers',
        title=f'{col} by Target Label',
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.update_layout(height=520, showlegend=False)
    fig.show()

## 8. Multivariate And Correlation Analysis

In [ ]:
corr, cov = multivariate_analysis(df, PLOTS_DIR)
corr_results = correlation_analysis(df)
pearson = corr_results['pearson']
spearman = corr_results['spearman']
kendall = corr_results['kendall']

if not pearson.empty:
    fig = px.imshow(
        pearson,
        text_auto='.2f' if pearson.shape[0] <= 14 else False,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Pearson Correlation Heatmap',
    )
    fig.update_layout(height=max(600, pearson.shape[0] * 32))
    fig.show()

strong_pos, strong_neg, weak = correlation_pairs(pearson)
display(Markdown('### Strongest Positive Correlations'))
display(strong_pos.round(4))
display(Markdown('### Strongest Negative Correlations'))
display(strong_neg.round(4))
display(Markdown('### Weak Relationships'))
display(weak.round(4))

In [ ]:
# Scatter matrix for a compact set of important numeric features
candidate_features = [
    'file_size_mb', 'duration_s', 'size_per_second_mb', 'bit_rate',
    'fps', 'width', 'height', 'aspect_ratio', 'pixels', 'token_count',
]
scatter_features = [c for c in candidate_features if c in df.columns and df[c].nunique(dropna=True) > 1][:6]
if len(scatter_features) >= 2:
    sample = df[scatter_features + (['label'] if 'label' in df else [])].dropna().sample(
        min(len(df.dropna(subset=scatter_features)), 1500),
        random_state=42,
    )
    fig = px.scatter_matrix(
        sample,
        dimensions=scatter_features,
        color='label' if 'label' in sample else None,
        title='Interactive Pairplot Matrix',
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.update_traces(diagonal_visible=False, showupperhalf=False)
    fig.update_layout(height=820)
    fig.show()

## 9. Target Variable Analysis

In [ ]:
target = target_analysis(df, PLOTS_DIR)
if target:
    target_table = pd.DataFrame({
        'count': target['class_distribution'],
        'percent': target['class_percent'],
    })
    display(target_table.round(4))
    display(Markdown(f"**Imbalance ratio:** {target.get('imbalance_ratio', np.nan):.4f}"))
    fig = px.pie(
        target_table.reset_index(names='label'),
        names='label',
        values='count',
        title='Target Class Distribution',
        hole=0.45,
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.update_layout(height=500)
    fig.show()
    if 'target_correlation' in target:
        display(Markdown('### Target Correlation Ranking'))
        display(target['target_correlation'].head(25).to_frame('correlation').round(4))
else:
    display(Markdown('No target label was inferred.'))

## 10. Data Quality Checks

In [ ]:
quality = data_quality_checks(df)
display(pd.DataFrame({
    'check': list(quality.keys()),
    'value': [str(v) for v in quality.values()],
}))

## 11. Statistical Tests

In [ ]:
tests = statistical_tests(df, config)
for test_name, table in tests.items():
    display(Markdown(f'### {test_name.replace("_", " ").title()}'))
    if table.empty:
        display(Markdown('_No applicable test results._'))
    else:
        display(table.round(6).head(40))
        table.to_csv(TABLES_DIR / f'{test_name}_tests.csv', index=False)

## 12. Feature Engineering And ML Readiness

In [ ]:
insights = feature_engineering_insights(df, numeric_summary, missing, outliers)
readiness = ml_readiness(df, missing, outliers, target, quality)

display(Markdown(f"### Model Readiness Score: **{readiness['model_readiness_score']}/100**"))
display(Markdown('### Risks'))
display(pd.Series(readiness['risks'] if readiness['risks'] else ['None detected'], name='risk').to_frame())
display(Markdown('### Feature Engineering Insights'))
display(pd.Series(insights, name='recommendation').to_frame())
display(Markdown('### Preprocessing Checklist'))
display(pd.Series(readiness['preprocessing_checklist'], name='task').to_frame())
display(Markdown('### Recommended Algorithms'))
display(pd.Series(readiness['recommended_algorithms'], name='algorithm').to_frame())

## 13. Export Full Markdown Report

In [ ]:
write_report(
    df=df,
    config=config,
    overview=overview,
    missing=missing,
    numeric_summary=numeric_summary,
    cat_tables=cat_tables,
    outliers=outliers,
    distribution_notes=distribution_notes,
    corr_results=corr_results,
    target=target,
    quality=quality,
    tests=tests,
    insights=insights,
    readiness=readiness,
)

display(Markdown(f"Full report exported to `{OUTPUT_DIR / 'EDA_REPORT.md'}`"))
display(Markdown(f"Manifest exported to `{OUTPUT_DIR / 'media_manifest.csv'}`"))